In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

In [3]:
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')



In [4]:
#make sure the join key of these tables are unique
orders    = orders.drop_duplicates(subset='order_id')
customers = customers.drop_duplicates(subset='customer_id')
sellers   = sellers.drop_duplicates(subset='seller_id')
products  = products.drop_duplicates(subset='product_id')

In [5]:
print(orders['order_id'].duplicated().sum())
print(items['order_id'].duplicated().sum())
print("->there are multiple items in one order")
print(customers['customer_id'].duplicated().sum())
print(sellers['seller_id'].duplicated().sum())
print(products['product_id'].duplicated().sum())
print(payments['order_id'].duplicated().sum())
print("->can use multiple payment methods for one order in one order")

0
13984
->there are multiple items in one order
0
0
0
4446
->can use multiple payment methods for one order in one order


In [6]:
dfs = {
    'orders':    orders,
    'items':     items,
    'customers': customers,
    'sellers':   sellers,
    'products':  products,
    'payments':  payments,
    'geolocation': geolocation
}

for name, df in dfs.items():
    null_count = df.isnull().sum()
    null_cols  = null_count[null_count > 0]
    
    if null_cols.empty:
        print(f"✅ {name}: no nulls")
    else:
        print(f"❌ {name}: has nulls")
        print(null_cols.to_string())
        print()

❌ orders: has nulls
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

✅ items: no nulls
✅ customers: no nulls
✅ sellers: no nulls
❌ products: has nulls
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2

✅ payments: no nulls
✅ geolocation: no nulls


In [7]:
#convert string date into datetime objects
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

#only keep delivered orders since we need both actual and estimate delivery time
orders = orders[orders['order_status'] == 'delivered'].copy()

#adding feature keeping track if an order is late or arrive early(positive= late, negative= early)
orders['delay_hours'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.total_seconds() / 3600
#merging features into time needed for approval (from purchase time to approved time)
orders['approval_time_hours'] = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600


#merging into time need to handle package to carrier from order placed
orders['carrier_time_hours'] = (orders['order_delivered_carrier_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600

#estimated delivery time from order placed
orders['estimated_delivery_hours'] = (orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600

#adding feature represent month and day of week purchase, some peak month, day might cause delay
orders['purchase_month']     = orders['order_purchase_timestamp'].dt.month
orders['purchase_dayofweek'] = orders['order_purchase_timestamp'].dt.dayofweek

In [8]:
# customer_unique_id identifies the real person across multiple orders
# customer_id is order-scoped — same person gets different customer_id each order

customer_order_counts = (
    customers.groupby('customer_unique_id')['customer_id']
    .count()
    .reset_index()
    .rename(columns={'customer_id': 'customer_order_count'})
)
customers = customers.merge(customer_order_counts, on='customer_unique_id', how='left')

In [9]:
# merge items per order to one row
items_agg = items.groupby('order_id').agg(
    num_items           = ('order_item_id', 'count'),
    total_price         = ('price', 'sum'),
    total_freight       = ('freight_value', 'sum'),
    seller_id           = ('seller_id', 'first'),
    shipping_limit_date = ('shipping_limit_date', 'first')
).reset_index()

In [10]:
# merge multiple methods of the same order to one row
# Clean 'not_defined' payment type from source data
# payment_methods_used: multiple methods may suggest order complexity

payments['payment_type'] = payments['payment_type'].replace('not_defined', 'unknown')
 
payments_agg = payments.groupby('order_id').agg(
    payment_installments = ('payment_installments', 'max'),
    payment_value        = ('payment_value', 'sum'),
    payment_type         = ('payment_type', 'first'),
    payment_methods_used = ('payment_sequential', 'max')
).reset_index()

In [11]:
products_clean = products[[
    'product_id', 'product_category_name',
    'product_weight_g', 'product_length_cm',
    'product_height_cm', 'product_width_cm',
    'product_photos_qty', 'product_name_lenght',
    'product_description_lenght'
]].copy()
 
items_with_products = items.merge(products_clean, on='product_id', how='left')
 
product_agg = items_with_products.groupby('order_id').agg(
    avg_weight             = ('product_weight_g', 'mean'),
    avg_length             = ('product_length_cm', 'mean'),
    avg_height             = ('product_height_cm', 'mean'),
    avg_width              = ('product_width_cm', 'mean'),
    product_category       = ('product_category_name', 'first')
).reset_index()
 
# Combine dimensions into single volume feature
product_agg['avg_volume_cm3'] = (
    product_agg['avg_length'] *
    product_agg['avg_height'] *
    product_agg['avg_width']
)
product_agg = product_agg.drop(columns=['avg_length', 'avg_height', 'avg_width'])

In [12]:
# Multiple lat/lon entries per zip prefix — average down to one row per zip
geo_agg = geolocation.groupby('geolocation_zip_code_prefix').agg(
    geo_lat = ('geolocation_lat', 'mean'),
    geo_lng = ('geolocation_lng', 'mean')
).reset_index()



In [13]:
#MERGE EVERYTHING INTO ONE TABLE
df = orders[[
    'order_id', 'customer_id', 'delay_hours',
    'approval_time_hours', 'carrier_time_hours', 'estimated_delivery_hours',
    'purchase_month', 'purchase_dayofweek',
    'order_approved_at','order_delivered_carrier_date'
]].copy()
 
df = df.merge(items_agg,    on='order_id',    how='left')
df = df.merge(payments_agg, on='order_id',    how='left')
df = df.merge(product_agg,  on='order_id',    how='left')
df = df.merge(
    customers[[
        'customer_id', 'customer_state',
        'customer_zip_code_prefix', 'customer_order_count'
    ]],
    on='customer_id', how='left'
)
df = df.merge(
    sellers[['seller_id', 'seller_state', 'seller_zip_code_prefix']],
    on='seller_id', how='left'
)
 
# Merge geolocation for customer zip
df = df.merge(
    geo_agg.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'geo_lat': 'customer_lat',
        'geo_lng': 'customer_lng'
    }),
    on='customer_zip_code_prefix', how='left'
)
 
# Merge geolocation for seller zip
df = df.merge(
    geo_agg.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
        'geo_lat': 'seller_lat',
        'geo_lng': 'seller_lng'
    }),
    on='seller_zip_code_prefix', how='left'
)
 

# same_state: same distance but different state might cause issue? so it worth to consider this even when has distance
# seller_shipping_window_hours: negative = seller already missed their deadline
# delivery_distance_km: actual km between seller and customer via Haversine

df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)
 
# Did the seller hand to carrier after the shipping limit deadline
# 1 = seller failed to ship on time, 0 = seller shipped on time
df['seller_missed_deadline'] = (df['order_delivered_carrier_date'] > df['shipping_limit_date']).astype(int)

#might be the reason of seller missing deadline
df['seller_shipping_window_hours'] = (df['shipping_limit_date'] - df['order_approved_at']).dt.total_seconds() / 3600

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlng  = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
 
df['delivery_distance_km'] = haversine(
    df['customer_lat'], df['customer_lng'],
    df['seller_lat'],   df['seller_lng']
)
#drop redundant features after engineering
df_clean= df.drop(columns=[
    'order_id', 'customer_id', 'seller_id',
    'order_approved_at', 'order_delivered_carrier_date', 'shipping_limit_date',
    'customer_state', 'seller_state',
    'customer_zip_code_prefix', 'seller_zip_code_prefix',
    'customer_lat', 'customer_lng',
    'seller_lat', 'seller_lng',
])

In [14]:

# Negative carrier time — physically impossible
print("\ncarrier_time_hours < 0 (impossible):")
negative_carrier = df_clean[df_clean['carrier_time_hours'] < 0]
print(f"  Count: {len(negative_carrier)}")
print(f"  % of total: {len(negative_carrier)/len(df_clean):.2%}")

# Negative estimated delivery — also impossible
print("\nestimated_delivery_hours < 0 (impossible):")
negative_estimated = df_clean[df_clean['estimated_delivery_hours'] < 0]
print(f"  Count: {len(negative_estimated)}")
print(f"  % of total: {len(negative_estimated)/len(df_clean):.2%}")


carrier_time_hours < 0 (impossible):
  Count: 165
  % of total: 0.17%

estimated_delivery_hours < 0 (impossible):
  Count: 0
  % of total: 0.00%


In [15]:
# Drop negative carrier_time — physically impossible, timestamp bug in system
df = df[df['carrier_time_hours'] >= 0]

#drop null delay hours
df_clean = df_clean.dropna(subset=['delay_hours'])

In [16]:
print(f"shape:  {df_clean.shape}")
print(df_clean.info())
print(f"\nMissing values:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"\ncolumns: {list(df_clean.columns)}")

shape:  (96470, 21)
<class 'pandas.DataFrame'>
Index: 96470 entries, 0 to 96477
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   delay_hours                   96470 non-null  float64
 1   approval_time_hours           96456 non-null  float64
 2   carrier_time_hours            96469 non-null  float64
 3   estimated_delivery_hours      96470 non-null  float64
 4   purchase_month                96470 non-null  int32  
 5   purchase_dayofweek            96470 non-null  int32  
 6   num_items                     96470 non-null  int64  
 7   total_price                   96470 non-null  float64
 8   total_freight                 96470 non-null  float64
 9   payment_installments          96469 non-null  float64
 10  payment_value                 96469 non-null  float64
 11  payment_type                  96469 non-null  str    
 12  payment_methods_used          96469 non-null  float64
 1

In [18]:
num_cols = df_clean.select_dtypes(include="number").columns
df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
cat_cols = df_clean.select_dtypes(include="object").columns
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
print(f"Shape after cleaning: {df_clean.shape}")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")
# Save training imputation values — reuse these on the test set to avoid leakage
train_num_medians = df_clean[num_cols].median()
train_cat_modes   = {col: df_clean[col].mode()[0] for col in cat_cols}

Shape after cleaning: (96470, 21)
Missing values remaining: 0


In [ ]:

# One-hot encode all categorical features
df_encoded = pd.get_dummies(df_clean, drop_first=False)

FEATURES = [c for c in df_encoded.columns if c not in ["delay_hours"]]
TARGET   = "delay_hours"

X = df_encoded[FEATURES]
y = df_encoded[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Features after one-hot encoding: {len(FEATURES)}")
print(f"Training rows: {X_train.shape[0]}")
print(f"Test rows:     {X_test.shape[0]}")

Features after one-hot encoding: 95
Training rows: 77176
Test rows:     19294
